# Pennsylvania Health, Food, Hospitals Web Mapping App Workflow

This notebook demonstrates a streaming GAS workflow that combines data retrieval, PASDA discovery, and web mapping app generation. It uses the published GAS client SDK directly and calls each agent with `agent.execute_task(..., mode="stream")`. The only helper function in this notebook extracts the first usable artifact URL from a completed GAS task response.

Workflow overview:

1. Use `geospatial_data_retrieval_agent` to download 2020 Pennsylvania county-level CDC PLACES adult obesity data.
2. Use `geospatial_data_retrieval_agent` to download Pennsylvania county boundaries.
3. Use `geospatial_data_retrieval_agent` to download Pennsylvania fast-food restaurant locations.
4. Use `pasda_agent` to download Pennsylvania hospital locations from PASDA.
5. Use `web_mapping_app_agent` to build a browser-ready HTML web mapping app that combines the county health data, county polygons, fast-food points, and hospital points.

All agent requests use streaming mode so progress events are visible while long-running model calls, data downloads, joins, and HTML generation are happening.


## Imports

In [ ]:
import sys
from pathlib import Path

from IPython.display import HTML, display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from gas_client import GasClient


## User Settings

Start the GAS server before running this notebook. For local development, keep `server_url` as `http://127.0.0.1:4042`. If you expose the server with a public URL such as ngrok, replace `server_url` with that public base URL.

The built-in agents used here are model-backed. Provide the credential accepted by your deployment. In this example, the OpenAI key is passed through the GAS client and included in each task request.


In [ ]:
server_url = "http://127.0.0.1:4042"
openai_api_key = "YOUR_OPENAI_API_KEY"
timeout_seconds = 1800

client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
    artifact_delivery="URL",
)

credentials = {
    "OPENAI_API_KEY": openai_api_key,
}


## Helper Function

This helper keeps the notebook cells focused on GAS operations. It only reads the standard `outputs.artifacts` section from a completed task response and returns a matching artifact URL.


In [ ]:
def first_artifact_url(task_result, preferred_extensions, task_label="task"):
    """Return the first artifact URL matching one of the preferred file extensions."""

    if not task_result:
        raise RuntimeError(f"The {task_label} did not return a task result.")

    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    extensions = tuple(ext.lower() for ext in preferred_extensions)

    for artifact in artifacts:
        if not isinstance(artifact, dict):
            continue

        url = artifact.get("url")
        if not url:
            continue

        candidates = [
            artifact.get("name"),
            artifact.get("filename"),
            url,
        ]
        for candidate in candidates:
            candidate_text = str(candidate or "").split("?", 1)[0].lower()
            if candidate_text.endswith(extensions):
                return url

    raise RuntimeError(f"The {task_label} did not return a usable artifact URL.")


## Discover Agents

In [ ]:
client.list_agents()


In [ ]:
data_agent = client.agent("geospatial_data_retrieval_agent")
pasda_agent = client.agent("pasda_agent")
web_mapping_app_agent = client.agent("web_mapping_app_agent")

for agent in [data_agent, pasda_agent, web_mapping_app_agent]:
    print(agent.agent_id, agent.status().get("status"))


## 1. Download 2020 Pennsylvania CDC Health Data And County Boundaries

The first two requests prepare the county-level inputs needed for choropleth mapping. The data retrieval agent first downloads CDC PLACES adult obesity data for Pennsylvania counties, then downloads Pennsylvania county boundaries. The web mapping app agent will later use these two artifacts together to create the county choropleth layer.


In [ ]:
cdc_health_result = None

for event in data_agent.execute_task(
    (
        "Download CDC PLACES county-level 2020 adult obesity prevalence "
        "for Pennsylvania counties. Save the result as a CSV file and "
        "include county FIPS/GEOID fields for joining to county boundaries."
    ),
    mode="stream",
    artifact_delivery="URL",
    credentials=credentials,
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        cdc_health_result = event.get("payload")

client.print_task_summary(cdc_health_result)

cdc_health_url = first_artifact_url(
    cdc_health_result,
    [".gpkg", ".geojson", ".json", ".csv"],
    task_label="CDC health retrieval task",
)

cdc_health_url


In [ ]:
pa_counties_result = None

for event in data_agent.execute_task(
    (
        "Download Pennsylvania county boundaries as a GeoPackage. "
    ),
    mode="stream",
    artifact_delivery="URL",
    credentials=credentials,
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        pa_counties_result = event.get("payload")

client.print_task_summary(pa_counties_result)

pa_counties_url = first_artifact_url(
    pa_counties_result,
    [".gpkg", ".geojson", ".json", ".zip"],
    task_label="Pennsylvania county boundary retrieval task",
)

pa_counties_url


## 2. Download Pennsylvania Fast-Food Restaurants

This request uses the data retrieval agent to retrieve fast-food restaurant point locations for Pennsylvania. These points will be displayed as an overlay in the final web mapping app.


In [ ]:
fast_food_result = None

for event in data_agent.execute_task(
    (
        "Download fast-food restaurant locations for Pennsylvania as a clean "
        "point dataset. Include useful name, brand, address, and category fields "
        "when available."
    ),
    mode="stream",
    artifact_delivery="URL",
    credentials=credentials,
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        fast_food_result = event.get("payload")

client.print_task_summary(fast_food_result)

fast_food_url = first_artifact_url(
    fast_food_result,
    [".gpkg", ".geojson", ".json"],
    task_label="fast-food retrieval task",
)

fast_food_url


## 3. Download Pennsylvania Hospitals From PASDA

This request uses the PASDA agent to find and download Pennsylvania hospital locations. The result will be used as another point layer in the web mapping app.


In [ ]:
hospitals_result = None

for event in pasda_agent.execute_task(
    (
        "Find and download the Pennsylvania hospitals dataset. Return a clean "
        "point dataset with hospital names and useful facility attributes."
    ),
    mode="stream",
    artifact_delivery="URL",
    credentials=credentials,
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        hospitals_result = event.get("payload")

client.print_task_summary(hospitals_result)

hospitals_url = first_artifact_url(
    hospitals_result,
    [".gpkg", ".geojson", ".json", ".zip"],
    task_label="PASDA hospitals task",
)

hospitals_url


## 4. Create The Web Mapping App

The web mapping app agent receives the downloaded artifact URLs and creates one browser-ready HTML application. The request asks the agent to join or visually combine the CDC health table with county polygons for a choropleth layer, add fast-food and hospital point layers, include popups, a real Leaflet/Folium layer control, compact legends, basemaps, a professional title, and an explanatory side panel.


In [ ]:
web_app_result = None

for event in web_mapping_app_agent.execute_task(
    (
        "Create a polished browser-ready Pennsylvania public health web mapping app. "
        "Use the CDC county-level obesity data and Pennsylvania county boundaries "
        "to create a county choropleth layer for adult obesity prevalence. The "
        "choropleth should use county polygons, a clear sequential color ramp, "
        "meaningful class breaks, and a compact bottom-right legend showing value "
        "ranges. Map fast-food restaurants and hospitals as separate point layers "
        "with distinct symbols or colors and informative popups. Include a "
        "professional title, an explanatory side panel, a real Leaflet/Folium layer "
        "control, useful basemaps, popups for all layers, clear symbology, and a "
        "Pennsylvania-focused map extent. Return the final result as one "
        "self-contained HTML file."
    ),
    mode="stream",
    input_datasets=[
        cdc_health_url,
        pa_counties_url,
        fast_food_url,
        hospitals_url,
    ],
    artifact_delivery="URL",
    credentials=credentials,
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        web_app_result = event.get("payload")

client.print_task_summary(web_app_result)

web_app_url = first_artifact_url(
    web_app_result,
    [".html"],
    task_label="web mapping app task",
)

web_app_url


## View The Web Mapping App

In [ ]:
from IPython.display import display, IFrame

display(IFrame(src=web_app_url, width="100%", height=760))

## Artifact URLs

In [ ]:
print("CDC health data:", cdc_health_url)
print("Fast-food restaurants:", fast_food_url)
print("PASDA hospitals:", hospitals_url)
print("Web mapping app:", web_app_url)
